# 📬 Notebook 1: Queues vs Pub/Sub — what's the difference?

When two pieces of software talk to each other, they often go through a **message broker** instead of calling each other directly. Two of the most common patterns are:

- 🟦 **Queue (point-to-point)** — one producer, many workers, and **only one** worker handles each message. Used for "do this job" workloads.
- 🟪 **Pub/Sub (topic)** — one publisher, many subscribers, and **every** subscriber gets a copy of every message. Used for "tell everyone this happened" workloads.

In this notebook we implement both with nothing but Python's standard library, so the mechanics are obvious.

## Learning objectives
- Feel the pain of **direct synchronous calls** and understand why we want a broker in the middle.
- Build a tiny in-memory queue and observe load-balancing across workers.
- Build a tiny in-memory pub/sub bus and observe fan-out to all subscribers.
- See what happens when a producer is faster than its consumers (**backpressure**).
- Know when to pick which.

## 🛠️ Setup

```bash
cd 01-foundations/messaging-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if the kernel doesn't appear: `Cmd+Shift+P` → **Reload Window**.

This notebook uses only Python's stdlib (`queue`, `threading`).

## ❌ Bad practice: direct synchronous calls

Imagine `OrderService` accepts an order and then has to:
1. send a confirmation email,
2. update the analytics dashboard,
3. notify the warehouse.

The naive way is to call each downstream service directly, in order. This works on a sunny day, but it has three sharp edges that bite hard in production:

1. **Tight coupling** — `OrderService` has to know every downstream's address. Adding a 4th listener (e.g., a fraud-detector) means *editing OrderService*.
2. **Cascading failure** — if any downstream is slow or down, the order request blocks or fails.
3. **No buffering** — a traffic spike hits every downstream at exactly the same time.

In [ ]:
import time

def email_service(order):
    time.sleep(0.05)                     # pretend network call
    print(f"  ✉️  emailed receipt for {order}")

def analytics_service(order):
    time.sleep(0.05)
    print(f"  📊 logged {order}")

def warehouse_service(order):
    raise RuntimeError("warehouse is down for maintenance")

def place_order_BAD(order):
    # OrderService directly calls every downstream — and waits for each one.
    email_service(order)
    analytics_service(order)
    warehouse_service(order)             # 💥 this one explodes
    print(f"  ✅ order {order} accepted")

start = time.perf_counter()
try:
    place_order_BAD("order-1")
except Exception as e:
    print(f"  ❌ order failed: {e}")
print(f"took {time.perf_counter() - start:.3f}s — and the customer never got their confirmation")

Notice three problems:
- The order **never completes** because the warehouse is down — even though emails and analytics already succeeded.
- `OrderService` paid the latency of *every* downstream sequentially.
- To add a new listener, we'd have to change this function.

The fix is to **put a broker in the middle**. The order service produces a message, and downstreams consume it on their own schedule. That's what the rest of this notebook builds.

## 🟦 Approach 1: Queue (one consumer per message)

Three workers share the same queue. Each message is delivered to **exactly one** of them — whichever pulls first wins. This naturally **load-balances** work.

In [ ]:
import queue
import threading
import time
import random

q = queue.Queue()
results = {}  # worker_id -> list of jobs it handled
results_lock = threading.Lock()

def worker(worker_id):
    while True:
        job = q.get()
        if job is None:        # poison pill = shutdown signal
            return
        time.sleep(random.uniform(0.01, 0.05))
        with results_lock:
            results.setdefault(worker_id, []).append(job)
        q.task_done()

# Start 3 workers
threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(3)]
for t in threads: t.start()

# Producer puts 10 jobs in the queue
for i in range(10):
    q.put(f"job-{i}")

q.join()  # wait for all jobs to be done
for _ in threads: q.put(None)  # tell workers to shut down
for t in threads: t.join()

for w, jobs in sorted(results.items()):
    print(f"worker {w} handled {len(jobs):2d} jobs: {jobs}")
print("\nNotice: each job was handled by exactly ONE worker.")

## 🟪 Approach 2: Pub/Sub (every subscriber gets a copy)

Now we want a different shape: when something interesting happens, we want **every** interested service to find out. Each subscriber gets its own private queue, and the publisher fans the message out to all of them.

In [ ]:
class PubSub:
    def __init__(self):
        self.subs: dict[str, list[queue.Queue]] = {}
        self._lock = threading.Lock()    # subs dict is touched by many threads

    def subscribe(self, topic: str) -> queue.Queue:
        inbox = queue.Queue()
        with self._lock:
            self.subs.setdefault(topic, []).append(inbox)
        return inbox

    def publish(self, topic: str, message):
        # Deliver a copy to EVERY subscriber on the topic.
        with self._lock:
            inboxes = list(self.subs.get(topic, []))
        for inbox in inboxes:
            inbox.put(message)

bus = PubSub()
analytics = bus.subscribe("user.signup")
welcome_email = bus.subscribe("user.signup")
audit_log = bus.subscribe("user.signup")

bus.publish("user.signup", {"user": "alice"})
bus.publish("user.signup", {"user": "bob"})

for name, inbox in [("analytics", analytics),
                    ("welcome_email", welcome_email),
                    ("audit_log", audit_log)]:
    received = []
    while not inbox.empty():
        received.append(inbox.get())
    print(f"{name:14s} got {received}")

print("\nNotice: every subscriber received EVERY message.")
print("Re-read the BAD example above — pub/sub is the decoupled version of it.")

## 🪣 Backpressure: what happens when the producer is too fast?

An unbounded queue grows without limit — until your process runs out of memory. Real brokers cap the queue size and either **block the producer** or **reject** new messages. Python's `queue.Queue(maxsize=N)` blocks the producer's `put()` once the queue is full, which is the simplest form of backpressure.

In [ ]:
import queue, threading, time

bounded = queue.Queue(maxsize=2)         # capacity = 2 messages

def slow_consumer():
    while True:
        item = bounded.get()
        if item is None: return
        time.sleep(0.1)                  # consumer takes 100 ms per message
        print(f"  consumed {item}")
        bounded.task_done()

threading.Thread(target=slow_consumer, daemon=True).start()

for i in range(5):
    t0 = time.perf_counter()
    bounded.put(f"msg-{i}")              # blocks once 2 messages are pending
    print(f"produced msg-{i} (waited {time.perf_counter() - t0:.2f}s)")

bounded.join()
bounded.put(None)
print("\nNotice: the producer was forced to slow down to match the consumer.")

## ⚠️ One word on ordering

It's tempting to assume "messages stay in the order I sent them." Be careful:

- A queue with **multiple workers** delivers messages roughly in order, but they *finish* out of order.
- **At-least-once retries** (next notebook) can re-deliver an old message after newer ones have been processed.
- Most brokers only guarantee order **per partition / per key** (e.g., "all events for user 42 stay in order").

## 🤔 When to use which?

| | Queue | Pub/Sub |
|---|---|---|
| Each message handled by | exactly one worker | every subscriber |
| Good for | background jobs, work distribution | event broadcasting |
| Examples | image resizing, sending emails | "user signed up", "order placed" |
| Real-world tools | SQS, RabbitMQ queues, Redis lists | Kafka topics, Redis pub/sub, NATS |

You will often see **both** in the same system: an event is *published*, and one of the subscribers is itself a queue feeding workers.